In [ ]:
%load_ext autoreload
%autoreload 2

from math import pi
from glob import glob

import numpy as np
import matplotlib.pyplot as plt

import ast

import pandas as pd

In [ ]:
from matplotlib.colors import LogNorm
plt.rcParams.update({'font.size': 18})

In [ ]:
from numpy.random import normal

In [ ]:
from scipy.optimize import curve_fit

### Generate data from a Gaussian distribution

In [ ]:
data = normal(0,1,1000)

In [ ]:
XMIN = -10
XMAX = 10
NBIN = 51

In [ ]:
# histogram the data
BINS = np.linspace(XMIN,XMAX,NBIN)
vals,edges = np.histogram(data,bins=BINS)
bincenters = 0.5*(edges[:-1]+edges[1:])
print(edges)
print(bincenters)
print(vals)
vals = vals.astype('float')
errs = np.sqrt(vals)

In [ ]:
fig = plt.figure(figsize=(6,6))
plt.hist(data,bins=BINS,histtype='step',lw=2)
plt.errorbar(bincenters,vals,yerr=errs,color='k',fmt='o')
plt.xlabel('value')
plt.ylabel('frequency')
plt.show()

In [ ]:
def Gaussian(x,mu,sigma,A):
    N = 1./(sigma * np.sqrt(2*np.pi))
    return A * N * np.exp(-0.5*((x-mu)**2)/(sigma**2))

### simple fit attempt

In [ ]:
popt,popv,infodict,mesg,ier = curve_fit(Gaussian,bincenters,vals,full_output=True)
print('function calls: ',infodict['nfev'])
fiterr = np.sqrt(np.diag(popv))
mu = popt[0]
mue = fiterr[0]
sigma = popt[1]
sigmae = fiterr[1]
A = popt[2]
Ae = fiterr[2]
print('mu \t = %.02f +/- %.02f'%(mu,mue))
print('sigma \t = %.02f +/- %.02f'%(sigma,sigmae))
print('A \t = %.02f +/- %.02f'%(A,Ae))

In [ ]:
fig = plt.figure(figsize=(6,6))
xvals = np.linspace(XMIN,XMAX,301)
LABEL = r'$\mu$ = %.02f +/- %.02f $\sigma$ = %.02f +/- %.02f'%(mu,mue,sigma,sigmae)
plt.plot(xvals,Gaussian(xvals,*popt),'r--',lw=2,label=LABEL)
plt.errorbar(bincenters,vals,yerr=errs,color='k',fmt='o')
plt.xlabel('value')
plt.ylabel('frequency')
plt.legend(loc=1,fontsize=12)
plt.ylim([0,plt.gca().get_ylim()[1]*1.2])
plt.show()

### use an initial guess

In [ ]:
GOODGUESS = [0,1,600] # good guess
BADGUESS = [-10,5,13] # bad guess
popt,popv,infodict,mesg,ier = curve_fit(Gaussian,bincenters,vals,full_output=True,p0=GOODGUESS)
print('function calls: ',infodict['nfev'])
fiterr = np.sqrt(np.diag(popv))
mu = popt[0]
mue = fiterr[0]
sigma = popt[1]
sigmae = fiterr[1]
A = popt[2]
Ae = fiterr[2]
print('mu \t = %.02f +/- %.02f'%(mu,mue))
print('sigma \t = %.02f +/- %.02f'%(sigma,sigmae))
print('A \t = %.02f +/- %.02f'%(A,Ae))

In [ ]:
fig = plt.figure(figsize=(6,6))
xvals = np.linspace(XMIN,XMAX,301)
LABEL = r'$\mu$ = %.02f +/- %.02f $\sigma$ = %.02f +/- %.02f'%(mu,mue,sigma,sigmae)
plt.plot(xvals,Gaussian(xvals,*popt),'r--',lw=2,label=LABEL)
plt.errorbar(bincenters,vals,yerr=errs,color='k',fmt='o')
plt.xlabel('value')
plt.ylabel('frequency')
plt.legend(loc=1,fontsize=12)
plt.ylim([0,plt.gca().get_ylim()[1]*1.2])
plt.show()

### include errors

In [ ]:
print(errs)
#condition = errs < 1.
#errs[condition] = 1.
print(errs)

popt,popv,infodict,mesg,ier = curve_fit(Gaussian,bincenters,vals,full_output=True,\
                                       absolute_sigma=True,\
                                       sigma=errs)
print('function calls: ',infodict['nfev'])
fiterr = np.sqrt(np.diag(popv))
mu = popt[0]
mue = fiterr[0]
sigma = popt[1]
sigmae = fiterr[1]
A = popt[2]
Ae = fiterr[2]
print('mu \t = %.02f +/- %.02f'%(mu,mue))
print('sigma \t = %.02f +/- %.02f'%(sigma,sigmae))
print('A \t = %.02f +/- %.02f'%(A,Ae))

In [ ]:
fig = plt.figure(figsize=(6,6))
xvals = np.linspace(XMIN,XMAX,301)
LABEL = r'$\mu$ = %.02f +/- %.02f $\sigma$ = %.02f +/- %.02f'%(mu,mue,sigma,sigmae)
plt.plot(xvals,Gaussian(xvals,*popt),'r--',lw=2,label=LABEL)
plt.errorbar(bincenters,vals,yerr=errs,color='k',fmt='o')
plt.xlabel('value')
plt.ylabel('frequency')
plt.legend(loc=1,fontsize=12)
plt.ylim([0,plt.gca().get_ylim()[1]*1.2])
plt.show()

### Check goodness of fit

In [ ]:
def ChiSq(binc,vals,errs,func,fitparam):
    chisq = 0.
    for i,b in enumerate(binc):
        v = vals[i]
        e = errs[i]
        f = func(b,*fitparam)
        chisq += ((f-v)**2/e**2)
    return chisq

In [ ]:
chi = ChiSq(bincenters,vals,errs,Gaussian,popt)
print('chisq is: %.02f'%chi)
print('Degrees of freedom: %i'%len(bincenters))